In [ ]:
# 1. Install required training libraries quietly
!pip install -q transformers[torch] datasets accelerate pandas huggingface_hub

import os
import torch
import pandas as pd
from huggingface_hub import login
from datasets import Dataset
from google.colab import drive # <--- NEW: Import Drive library
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# 2. Mount Google Drive (This will pop up a permission window)
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# 3. Authenticate Hugging Face
hf_token = os.environ.get("HF_TOKEN", "")  # Set in Colab secrets or environment
if hf_token:
    login(token=hf_token)

# 4. Load and Merge Data
print("Loading and merging datasets...")
train_df = pd.read_csv("train.csv")
complaints_df = pd.read_csv("chief_complaints.csv")

# Merge on patient_id so every row has both text and a label
df = pd.merge(train_df, complaints_df, on="patient_id")

TEXT_COLUMN = "chief_complaint_raw"
LABEL_COLUMN = "triage_acuity"
MODEL_NAME = "Yuvrajxms09/biobert-triage-classifier"

# <--- NEW: Set the output directory to your personal Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/fine_tuned_biobert_triage"

# 5. Preprocess the Data
df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)

# Convert labels to categorical integers
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype('category')
label_mapping = dict(enumerate(df[LABEL_COLUMN].cat.categories))
df['label'] = df[LABEL_COLUMN].cat.codes
num_labels = len(label_mapping)

print(f"Detected {num_labels} distinct triage classes: {label_mapping}")

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df[[TEXT_COLUMN, 'label']].rename(columns={TEXT_COLUMN: 'text'}))

# 6. Tokenize Data
print("\nLoading tokenizer and tokenizing data...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)

# 7. Load the Model
print("\nLoading pre-trained BioBERT model weights...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)

# 8. Define Training Arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
)

# 9. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
)

# 10. Execute Training
print("\nStarting model training...")
trainer.train()

# 11. Save the final model directly to Google Drive
print(f"\nTraining complete! Saving fine-tuned model directly to: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Model saved successfully to your Google Drive! You can safely close Colab now without losing your work.")

Connecting to Google Drive...
Mounted at /content/drive
Loading and merging datasets...
Detected 5 distinct triage classes: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5}

Loading tokenizer and tokenizing data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Training complete! Saving fine-tuned model directly to: /content/drive/MyDrive/fine_tuned_biobert_triage


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully to your Google Drive! You can safely close Colab now without losing your work.


In [6]:
# 1. Install necessary libraries
!pip install -q transformers torch pandas

import torch
import pandas as pd
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from IPython.display import display

# 2. Connect to Google Drive
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# 3. Define file paths (ALL pointing to your permanent Google Drive now)
MODEL_PATH = "/content/drive/MyDrive/fine_tuned_biobert_triage"
COMPLAINTS_FILE = "/content/drive/MyDrive/fine_tuned_biobert_triage/chief_complaints.csv"
OUTPUT_FILE = "/content/drive/MyDrive/fine_tuned_biobert_triage/triage_predictions_results.csv"

# 4. Load the fine-tuned model and tokenizer
print("\nLoading your custom BioBERT model from Google Drive...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Set up the inference pipeline, forcing it to use the GPU (device=0) if available
device = 0 if torch.cuda.is_available() else -1
triage_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device
)

# 5. Load the new data
print("\nLoading chief complaints data...")
df = pd.read_csv(COMPLAINTS_FILE)
text_column = "chief_complaint_raw" # Make sure this matches your CSV

# Drop empty rows just in case
df = df.dropna(subset=[text_column])

# 6. Run Batch Predictions (OPTIMIZED FOR GPU)
print(f"\nRunning predictions on {len(df)} records using batched GPU processing...")

# Convert all text to a standard Python list (much faster for the pipeline)
texts = df[text_column].astype(str).tolist()

# Pass the entire list to the pipeline and set batch_size=64
# This forces the GPU to process 64 records at the exact same time
predictions = triage_pipe(texts, batch_size=64, truncation=True, max_length=128)

# Extract the labels and scores from the results
clean_labels = [res['label'].replace('LABEL_', '') for res in predictions]
confidence_scores = [round(res['score'], 4) for res in predictions]

# Assign them back to your dataframe
df['predicted_triage_level'] = clean_labels
df['confidence_score'] = confidence_scores

print("\nPredictions Complete! Previewing the first few results:")
display(df[[text_column, 'predicted_triage_level', 'confidence_score']].head())

# 7. Save directly to Google Drive
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSUCCESS! Results permanently saved to your Google Drive at: {OUTPUT_FILE}")

Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Loading your custom BioBERT model from Google Drive...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading chief complaints data...

Running predictions on 100000 records using batched GPU processing...

Predictions Complete! Previewing the first few results:


,chief_complaint_raw,predicted_triage_level,confidence_score
0,"thunderclap headache, worsening with movement",1,1.0
1,"contraception advice, intermittent",4,1.0
2,"general health question, intermittent",4,1.0
3,"erythema migrans tick bite, intermittent",2,1.0
4,"cellulitis localised, intermittent",2,1.0



SUCCESS! Results permanently saved to your Google Drive at: /content/drive/MyDrive/fine_tuned_biobert_triage/triage_predictions_results.csv
